In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

import seaborn as sns
from helper_functions import *

## Pre-process the decision variables

In [ ]:
# create decision variable files with headers 
dv_names = ['RT_W', 'RT_D', 'RT_F', 'TT_D', 'TT_F', 'LMA_W', 'LMA_D', 'LMA_F', \
    'AP_W', 'AP_D', 'AP_F', 'IT_W', 'IT_D', 'IT_F', 'IP_W', 'IP_D', 'IP_F', \
    'INF_W', 'INF_D', 'INF_F', 'NRR_W', 'CRL', 'CRH', 'WR1', 'SCR', 'GQR', \
    'NRR_F', 'WR2_W', 'WR2_D', 'WR2_F']

dv_base_filename = 'objectives_dvs_nd/dvs_base_nd_truncated_extremes.csv'
dv_iu_filename = 'objectives_dvs_nd/dvs_IU_nd_truncated_extremes.csv'

trigger_names = ['RT_W', 'RT_D', 'RT_F', 'TT_D', 'TT_F','INF_W', 'INF_D', 'INF_F']
alloc_names = ['LMA_W', 'LMA_D', 'LMA_F', 'AP_W', 'AP_D', 'AP_F']
dvs_relevant = trigger_names + alloc_names
infra_W = ['NRR_W', 'CRL', 'CRH', 'WR1', 'GQR', 'WR2_W']
infra_D = ['SCR', 'WR2_D']
infra_F = ['NRR_F', 'WR2_F']

util_abbrevs = ['W', 'D', 'F']
util_names_dict = {'W': 'Watertown', 'D': 'Dryville', 'F': 'Fallsland'}

num_dvs = len(dv_names)
print('Number of decision variables: ', num_dvs)
# import decision variable files 
dv_iu = pd.read_csv(dv_iu_filename, index_col=None, header=None)
dv_base = pd.read_csv(dv_base_filename, index_col=None, header=None)

# replace column headers with abbreviated names
dv_iu.columns = dv_names
dv_base.columns = dv_names

dv_IU_triggers = dv_iu[trigger_names]
dv_base_triggers = dv_base[trigger_names]
dvs_base_relevant = dv_base[dvs_relevant]
dvs_IU_relevant = dv_iu[dvs_relevant]
infra_W_triggers_IU = dv_iu[infra_W]
infra_D_triggers_IU = dv_iu[infra_D]   
infra_F_triggers_IU = dv_iu[infra_F]
infra_W_triggers_base = dv_base[infra_W]
infra_D_triggers_base = dv_base[infra_D]
infra_F_triggers_base = dv_base[infra_F]

## Select the objective mode you are working in

In [ ]:
obj_mode = "p90"

## Pull robustness data

In [ ]:
robustness_IU_dir = f"../robustness_analysis/robustness_sim_{obj_mode}_refset_IU.csv"
robustness_base_dir = f"../robustness_analysis/robustness_sim_{obj_mode}_refset_base.csv"

robustness_IU = pd.read_csv(robustness_IU_dir, index_col=None, header=0)
robustness_base = pd.read_csv(robustness_base_dir, index_col=None, header=0)

# get all robustness IU values that are higher than base 
robustness_R_base_min = robustness_base[['Regional']].min()
robustness_W_base_min = robustness_base[['Watertown']].min()
robustness_D_base_min = robustness_base[['Dryville']].min()
robustness_F_base_min = robustness_base[['Fallsland']].min()

robustness_IU_filtered_R = robustness_IU[robustness_IU['Regional'] > robustness_R_base_min.values[0]]
robustness_IU_filtered_W = robustness_IU[robustness_IU['Watertown'] > robustness_W_base_min.values[0]]
robustness_IU_filtered_D = robustness_IU[robustness_IU['Dryville'] > robustness_D_base_min.values[0]]
robustness_IU_filtered_F = robustness_IU[robustness_IU['Fallsland'] > robustness_F_base_min.values[0]]

# keep only the solutions that are more robust than base across all utilities
solutions_IU_morerobust = robustness_IU_filtered_R.index.intersection(robustness_IU_filtered_W.index)\
    .intersection(robustness_IU_filtered_D.index).intersection(robustness_IU_filtered_F.index)

# save IU solutions that are more robust than base 
np.savetxt(f'../solution_coop_ids/solutions_IU_morerobust_{obj_mode}.csv', solutions_IU_morerobust, fmt='%d')


In [ ]:
solutions_IU_morerobust.shape

In [ ]:
dv_IU_triggers_morerobust = dv_IU_triggers.loc[solutions_IU_morerobust]
dvs_IU_relevant_morerobust = dvs_IU_relevant.loc[solutions_IU_morerobust]

## Plot DV distributions

In [ ]:
sol_high_coop_IU = find_sols(dv_IU_triggers, "high_coop", [0.5])
sol_low_coop_IU = find_sols(dv_IU_triggers, "low_coop", [0.5])

sol_high_coop_base = find_sols(dv_base, "high_coop", [0.5])
sol_low_coop_base = find_sols(dv_base, "low_coop", [0.5])

#color_dict_coop = {'high_coop': '#20A39E', 'med_coop': "#FFBA49", 'low_coop': '#EF5B5B'}
color_dict_coop_IU = {'high_coop': '#F1A45D', 'low_coop': '#F8D8C2', 'others': '#D3D3D3'}
color_dict_coop_base = {'high_coop': '#6B8F71', 'low_coop': '#A3B18A', 'others': '#D3D3D3'}

In [ ]:
# save solution dataframes to csv
np.savetxt(f'../solution_coop_ids/sol_high_coop_IU.csv', sol_high_coop_IU, fmt='%d')
np.savetxt(f'../solution_coop_ids/sol_low_coop_IU.csv', sol_low_coop_IU, fmt='%d')
np.savetxt(f'../solution_coop_ids/sol_high_coop_base.csv', sol_high_coop_base, fmt='%d')
np.savetxt(f'../solution_coop_ids/sol_low_coop_base.csv', sol_low_coop_base, fmt='%d')

In [ ]:
np.array(sol_high_coop_IU).shape

## Plot DV polar plots

In [ ]:
#sols_IU = [sol_high_coop_IU[0], sol_med_coop_IU[6], sol_low_coop_IU[0]]
#sols_IU = [570, 429, sol_low_coop_IU[0]]  #571
#sols_IU = [268, 307]#
sols_IU = [558, 228]   #[129, 191, 201, 331, 587, 497, 178, 563, 339, 501, 475, 535, 502, 90, 91, 543]
sols_IU_selected = {'low_coop': sols_IU[1], 'high_coop': sols_IU[0]}
dv_names = ['RT', 'TT', 'INF', 'LMA' , 'AP']
fig, axs = plt.subplots(1, 3, figsize=(12, 6), subplot_kw={'projection': 'polar'})
for i, utility in enumerate(['Watertown', 'Dryville', 'Fallsland']):
    if utility == 'Watertown':
        dv_names = ['RT', 'INF', 'LMA' , 'AP']
        ax = axs[i]
        plot_polar_selected_dvs(dvs_IU_relevant, sols_IU, color_dict_coop_IU, dv_names, utility, ax)
        # plot a horizontal gridline at radius = 0.5
        ax.axhline(y=0.5, color='k', linestyle='--', linewidth=1.0)
    else:
        dv_names = ['RT', 'TT', 'INF', 'LMA' , 'AP']
        ax = axs[i]
        plot_polar_selected_dvs(dvs_IU_relevant, sols_IU, color_dict_coop_IU, dv_names, utility, ax)

# Move legend from center of the first subplot to the center of the main figure
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center', bbox_to_anchor=(0.5, 0.15), ncol=2, frameon=False)
plt.tight_layout()
plt.savefig('figures/polar_dvs_IU_selectedsols.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#sols_IU_families = {'low_coop': sol_low_coop_IU, 'high_coop': sol_high_coop_IU}
sols_IU_families = {'high_coop': sol_high_coop_IU, 'low_coop': sol_low_coop_IU}
utils_list = ['Watertown', 'Dryville', 'Fallsland']
fig, axs = plt.subplots(3, 1, figsize=(6,12), subplot_kw={'projection': 'polar'})
axs = axs.flatten()
for i, utility in enumerate(utils_list):
    if utility == 'Watertown':
        dv_names = ['RT', 'INF', 'LMA' , 'AP']
        ax = axs[i]
        # decision_vars_all, dvs_selected, dvs_color_dict, dv_names, utility, ax
        plot_polar_families(dvs_IU_relevant, sols_IU_families, color_dict_coop_IU, dv_names, utility, ax)
    else:
        dv_names = ['RT', 'TT', 'INF', 'LMA' , 'AP']
        ax = axs[i]
        plot_polar_families(dvs_IU_relevant, sols_IU_families, color_dict_coop_IU, dv_names, utility, ax)

# Move legend from center of the first subplot to the center of the main figure
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center', bbox_to_anchor=(0.5, -0.05), ncol=4, frameon=False)
plt.tight_layout()
plt.savefig('figures/polar_dvs_IU_families.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#sols_base_families = {'low_coop': sol_low_coop_base, 'high_coop': sol_high_coop_base}
sols_base_families = {'high_coop': sol_high_coop_base}
utils_list = ['Watertown', 'Dryville', 'Fallsland']
fig, axs = plt.subplots(3, 1, figsize=(6, 12), subplot_kw={'projection': 'polar'})
axs = axs.flatten()
for i, utility in enumerate(utils_list):
    if utility == 'Watertown':
        dv_names = ['RT', 'INF', 'LMA' , 'AP']
        ax = axs[i]
        # decision_vars_all, dvs_selected, dvs_color_dict, dv_names, utility, ax
        plot_polar_families(dvs_base_relevant, sols_base_families, color_dict_coop_base, dv_names, utility, ax)
    else:
        dv_names = ['RT', 'TT', 'INF', 'LMA' , 'AP']
        ax = axs[i]
        plot_polar_families(dvs_base_relevant, sols_base_families, color_dict_coop_base, dv_names, utility, ax)
    
# Move legend from center of the first subplot to the center of the main figure
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center', bbox_to_anchor=(0.5, 0.-0.05), ncol=4, frameon=False)
plt.tight_layout()
plt.savefig('figures/polar_dvs_base_families.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#12, 10, 18
#sols_base = [sol_high_coop_base[0], sol_med_coop_base[19], sol_low_coop_base[0]]
#sols_base = [28, 31]   [36, 43]
sols_base = [36, 31]
utils_list = ['Watertown', 'Dryville', 'Fallsland']
fig, axs = plt.subplots(1, 3, figsize=(12, 6), subplot_kw={'projection': 'polar'})
for i, utility in enumerate(utils_list):
    if utility == 'Watertown':
        dv_names = ['RT', 'INF', 'LMA' , 'AP']
        ax = axs[i]
        plot_polar_selected_dvs(dvs_base_relevant, sols_base, color_dict_coop, dv_names, utility, ax)
    else:
        dv_names = ['RT', 'TT', 'INF', 'LMA' , 'AP']
        ax = axs[i]
        plot_polar_selected_dvs(dvs_base_relevant, sols_base, color_dict_coop, dv_names, utility, ax)

# Move legend from center of the first subplot to the center of the main figure
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center', bbox_to_anchor=(0.5, 0.15), ncol=4, frameon=False)
plt.tight_layout()
plt.savefig('figures/polar_dvs_base_selectedsols.png', dpi=300, bbox_inches='tight')
plt.show()

### Find tradeoffs that correspond to the solutions

In [ ]:
obj_names = ['REL', 'RF', 'INPC', 'PFC', 'WCC']
util_abbrevs = ['W', 'D', 'F']

# append obj names to util_abbrevs
objs_utils = [f'{obj}_{util}' for util in util_abbrevs for obj in obj_names]
objs_regional =  [f'{obj}_R' for obj in obj_names]
objs_W = [f'{obj}_W' for obj in obj_names]
objs_D = [f'{obj}_D' for obj in obj_names]
objs_F = [f'{obj}_F' for obj in obj_names]

In [ ]:
objs_base = pd.read_csv(f'objectives_dvs_nd/objectives_base_p10_nd.csv', index_col=False, header=None)
objs_IU = pd.read_csv(f'objectives_dvs_nd/objectives_IU_p10_nd.csv', index_col=False, header=None)

objs_base.columns = objs_utils
objs_IU.columns = objs_utils

objs_base_W = objs_base[objs_W]
objs_base_D = objs_base[objs_D]
objs_base_F = objs_base[objs_F]
objs_avg_dict = {'Watertown': objs_base_W, 'Dryville': objs_base_D, 'Fallsland': objs_base_F}

objs_IU_W = objs_IU[objs_W]
objs_IU_D = objs_IU[objs_D]
objs_IU_F = objs_IU[objs_F]
objs_IU_dict = {'Watertown': objs_IU_W, 'Dryville': objs_IU_D, 'Fallsland': objs_IU_F}

objs_base_reg = find_regional_minimax(objs_base, obj_names)
objs_IU_reg = find_regional_minimax(objs_IU, obj_names)

In [ ]:
from matplotlib import colormaps, cm
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from pandas.plotting import parallel_coordinates
from parallel_plot_functions import *

In [ ]:
obj_mode = 'IU'
utility = 'Regional'
obj_util = objs_IU_reg.copy()
sols_selected = sols_IU
#obj_util = objs_IU_dict[utility].copy()

obj_cat = obj_util.copy()
obj_cat['Category'] = f'{obj_mode}_solution'
obj_high_coop = pd.DataFrame(obj_util.iloc[sols_selected[0], :]).T
#obj_high_coop = pd.DataFrame(obj_util.iloc[sols_IU_selected['high_coop'], :]).T
obj_high_coop['Category'] = 'high_coop'
obj_low_coop = pd.DataFrame(obj_util.iloc[sols_selected[1], :]).T
#obj_low_coop = pd.DataFrame(obj_util.iloc[sols_IU_selected['low_coop'], :]).T
obj_low_coop['Category'] = 'low_coop'

objs_selected = pd.concat([obj_high_coop, obj_low_coop], axis=0)
objs_parallel = pd.concat([obj_cat, obj_high_coop, obj_low_coop], axis=0)

parallel_color_dict = {f'{obj_mode}_solution': "#BBBBBB", 'high_coop': '#33658A', 'low_coop': '#86BBD8'}
obj_high_coop

In [ ]:
# setup figure specifications and dimensions
figsize = (10,5)
fontsize = 14
axis_mins = [0.90, 0.0, 0.0, 0.0, 0.0]
axis_maxs = [1.0, 1.0, 300, 1.0, 1.0]

# cap axis mins and maxs to reasonable values
axis_mins = np.maximum(axis_mins, 0.0)
axis_maxs[0] = np.minimum(axis_maxs[0], 1.0)
# get the axis labels 
axis_labels = obj_cat.columns.tolist()[:-1]
axis_labels

In [ ]:

### create figure
fig, ax = plt.subplots(1,1,figsize=figsize, gridspec_kw={'hspace':0.1, 'wspace':0.1})
custom_parallel_coordinates(fig, ax, objs_parallel, columns_axes=axis_labels, 
                                axis_labels=axis_labels, 
                                ideal_direction='bottom',
                                alpha_base=0.7, lw_base=2.5, fontsize=fontsize,
                                minmaxs=['max','min','min','min','min'], 
                                colorbar_ticks_continuous=range(0,1,5),
                                color_by_categorical = 'Category', 
                                color_dict_categorical=parallel_color_dict,
                                axis_mins=axis_mins, axis_maxs=axis_maxs) 
custom_parallel_coordinates(fig, ax, objs_selected, columns_axes=axis_labels, 
                                axis_labels=axis_labels, 
                                ideal_direction='bottom',
                                alpha_base=0.7, lw_base=4, fontsize=fontsize,
                                minmaxs=['max','min','min','min','min'], 
                            colorbar_ticks_continuous=range(0,1,5),
                                color_by_categorical = 'Category', 
                                color_dict_categorical=parallel_color_dict,
                                axis_mins=axis_mins, axis_maxs=axis_maxs) 

ax.set_title(f'Selected solutions for the region ({obj_mode}-avg ref. set)', fontsize=fontsize+2, pad=20)
plt.savefig(f'figures/parallel_selected_{obj_mode}_regional_nd.png', dpi=300, bbox_inches='tight')

In [ ]:
# among all IU solutions, find ones that are closest to the base solutions
sol_num_IU = sol_high_coop_IU
min_dist = float('inf')
sol_closest_high = sol_num_IU[0]
for sol in sol_high_coop_IU:
    dist = np.linalg.norm(objs_base_reg.values - objs_IU_reg.iloc[sol, :].values)
    if dist < min_dist:
        min_dist = dist
        sol_closest_high = sol

In [ ]:
sol_closest_high